# PDAN8412/w — Part 2: Bestseller Classification with Logistic Regression from Scratch
**Student:** Andiswa Sibusisiwe Phewa (ST10123828).  \n**Data analyst brief:** Evaluate publicly available book-market records for a fictional book press and develop an independently implemented binary classifier. **Dataset:** Amazon Kindle Books 2023 (`kindle_data-v2.csv`). **Target:** `isBestSeller` (snapshot status, *not* verified future sales).  
**Study design:** stratified 70/15/15 train/validation/test; training-only preprocessing; weighted and unweighted gradient descent; validation-selected threshold; one final test evaluation.  
**Academic integrity:** Original independently written NumPy model; external source and library documentation attributed in closing references. Results are outputs from execution rather than estimates.

**Critical interpretation:** An observed Amazon bestseller flag is not proof that the same title will become a future bestseller. The dataset contains contemporaneous market indicators; features observed only after commercial success are **excluded from the forward-looking model**. A deployment-ready prospective predictor would require point-in-time labels and features.

## 1. Dataset justification and alternatives (10 marks)
The open 2023 Kindle dataset has **133,102** rows (as verified below), 16 raw columns and an existing boolean bestseller label. This exceeds the 10,000-record minimum. It supports genuine *snapshot classification* without manufacturing labels from review scores. Important concerns are extreme target imbalance, undated bestseller-label provenance, unavailable sales totals, missing publication dates, platform coverage and differences between Amazon bestseller flags and company-specific commercial outcomes. 

**Alternatives screened conceptually:** (1) Goodreads book metadata/ratings: ratings may be available but no direct bestseller label; (2) Book-Crossing book-ratings data: abundant interactions but no verified bestseller label. The Kindle dataset is preferred because an explicit target exists. These alternatives were not used in the modelling and no unverified counts are assigned to them.

**Source:** asaniczka (2023), *Amazon Kindle Books Dataset 2023 – 130K Books*, Kaggle, https://www.kaggle.com/datasets/asaniczka/amazon-kindle-books-dataset-2023-130k-books . Source year denotes the dataset title / 2023 snapshot; the exact extraction date is not inferred.

**Forecasting caveat:** Our retained features are book listing descriptors rather than post-success engagement metrics; this reduces leakage but does not turn a cross-sectional snapshot into a genuine release-time prediction dataset.

## 2. Compact analysis plan (20 marks)
| Step | Method | Decision rule / expected output |
|---|---|---|
| EDA & quality | Spark `read.csv`, schema, nulls, label rates; Python figures | Inspect zeros, dates, duplicates, prevalence and distributions |
| Feature choice | Exclude identifiers, review counts, star ratings, editor awards and URLs | Prefer price, category, title length, announced publication year and Kindle programme indicator |
| Split & prepare | Stratified train/validation/test; fit imputers, scaler, encoder on train | Preserve prevalence and prevent leakage |
| Implement | NumPy sigmoid, BCE loss, gradients, batch GD, weights, L2 | No library logistic model for fitting |
| Improve | Compare unweighted vs class-weighted / regularised fits | Select configuration and threshold using validation **F1**; also review AP |
| Evaluate | Test once; confusion matrix, precision/recall/F1, ROC-AUC/AP | Interpret false positives/negatives and class imbalance |
| Report | Concise figures, model outcomes and limitations | Business recommendations with appropriate caution |

**Research logic:** Stratification preserves rare positive observations across splits; the test set is untouched until the last evaluation. Sampling remains at book level because each ASIN is unique in the inspected data.

In [1]:
from pathlib import Path
import json, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
from scipy.special import expit

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA_FILE = ROOT / 'kindle_data' / 'kindle_data-v2.csv'
OUT = ROOT / 'part2_outputs'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
for directory in (FIG, TAB): directory.mkdir(parents=True, exist_ok=True)
assert DATA_FILE.exists(), f'Missing dataset: {DATA_FILE}. Set Jupyter working directory to project root.'
print('Data:', DATA_FILE, ' | Python:', platform.python_version())

Data: /mnt/data/PDAN8412_Part2_ST10123828_REVISED/kindle_data/kindle_data-v2.csv  | Python: 3.13.5


## 3. Spark EDA and environment disclosure
**Compulsory Spark analysis code** follows. If PySpark is installed and Java is configured, these operations execute on the complete CSV. In a limited environment where PySpark cannot be installed, the notebook **explicitly reports non-execution**, then runs equivalent pandas summaries so the rest of the independent modelling remains reproducible. **Run the Spark cell in an environment with PySpark before final assessment submission; a fallback is not evidence that Spark ran.**

In [2]:
SPARK_EXECUTED = False
SPARK_STATUS = 'NOT RUN'
try:
    from pyspark.sql import SparkSession, functions as F
    spark = (SparkSession.builder.master('local[*]').appName('PDAN8412_Part2_KindleEDA')
             .config('spark.sql.shuffle.partitions', '8').getOrCreate())
    sdf = spark.read.option('header', True).option('inferSchema', True).csv(str(DATA_FILE))
    print('Spark schema:'); sdf.printSchema()
    n_spark = sdf.count()
    print('Spark records:', n_spark)
    print('Spark label frequencies:')
    sdf.groupBy('isBestSeller').count().orderBy('isBestSeller').show()
    print('Spark missing values:')
    sdf.select([F.sum(F.col(col).isNull().cast('int')).alias(col) for col in sdf.columns]).show(truncate=False)
    print('Spark categories by bestseller rate:')
    sdf.groupBy('category_name').agg(F.count('*').alias('books'),
      F.avg(F.col('isBestSeller').cast('int')).alias('best_rate')).orderBy(F.desc('best_rate')).show(10, truncate=False)
    print('Spark duplicate ASIN count:', n_spark - sdf.select('asin').distinct().count())
    SPARK_EXECUTED = True
    SPARK_STATUS = 'EXECUTED SUCCESSFULLY'
    spark.stop()
except Exception as exc:
    SPARK_STATUS = f'NOT EXECUTED: {type(exc).__name__}: {str(exc)[:200]}'
    print('SPARK DISCLOSURE:', SPARK_STATUS)
    print('Install and rerun with: pip install pyspark>=3.5,<5; configure Java (JDK).')
print('Spark completion status:', SPARK_STATUS)

SPARK DISCLOSURE: NOT EXECUTED: ModuleNotFoundError: No module named 'pyspark'
Install and rerun with: pip install pyspark>=3.5,<5; configure Java (JDK).
Spark completion status: NOT EXECUTED: ModuleNotFoundError: No module named 'pyspark'


## 4. Dataset loading, quality checks and EDA (20 marks)
Python inspection below is real execution. Zero star scores and reviews may include unavailable observations, so they should not automatically be interpreted as genuine consumer dissatisfaction or zero popularity. The class prevalence determines evaluation choices.

In [3]:
raw = pd.read_csv(DATA_FILE, low_memory=False)
assert {'asin','isBestSeller','title','price','category_name'}.issubset(raw.columns)
assert raw['isBestSeller'].dropna().isin([True,False]).all()
quality = pd.DataFrame({'dtype':raw.dtypes.astype(str),'missing':raw.isna().sum(),
                        'missing_pct':(raw.isna().mean()*100).round(2)})
quality.to_csv(TAB/'01_data_quality.csv')
label_counts = raw['isBestSeller'].value_counts().sort_index()
positive_rate = float(raw['isBestSeller'].mean())
print('Shape:',raw.shape,'| positive cases:',label_counts.get(True,0),
      '| prevalence:',f'{positive_rate:.3%}')
print('Exact duplicate rows:',raw.duplicated().sum(),
      '| duplicate ASINs:',raw['asin'].duplicated().sum())
print('Top missing fields:'); display(quality.sort_values('missing',ascending=False).head(8))
print('Numeric summaries:'); display(raw[['price','stars','reviews']].describe().round(2))
quality.to_csv(TAB/'01_data_quality.csv')
assert len(raw)>=10000 and raw.asin.is_unique

Shape: (133102, 16) | positive cases: 2236 | prevalence: 1.680%
Exact duplicate rows: 0 | duplicate ASINs: 0
Top missing fields:


,dtype,missing,missing_pct
publishedDate,object,49016,36.83
soldBy,object,9233,6.94
author,object,425,0.32
title,object,0,0.00
imgUrl,object,0,0.00
productURL,object,0,0.00
stars,float64,0,0.00
asin,object,0,0.00


Numeric summaries:


,price,stars,reviews
count,133102.00,133102.00,133102.00
mean,15.13,4.40,887.38
std,22.25,0.75,5104.88
min,0.00,0.00,0.00
25%,4.99,4.40,0.00
50%,9.99,4.50,4.00
75%,14.99,4.70,365.00
max,682.00,5.00,618227.00


In [4]:
# An intentionally restrained editorial colour scheme, no gradients or unrelated illustrations.
INK='#242923'; FOREST='#183A32'; OLIVE='#8A9275'; STONE='#D8D1C5'; RED='#9E4F3B'
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','font.size':10,
  'axes.spines.top':False,'axes.spines.right':False,'axes.labelcolor':INK,'text.color':INK,
  'axes.titleweight':'semibold','savefig.dpi':160})
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/name,bbox_inches='tight'); plt.close()

fig, ax=plt.subplots(figsize=(7.3,3.8))
vals=[int(label_counts.get(False,0)),int(label_counts.get(True,0))]
ax.bar(['Not bestseller','Bestseller'], vals, color=[STONE,FOREST],width=.55)
ax.set_title('Figure 1. Bestseller status is highly imbalanced');ax.set_ylabel('Number of books')
for i,v in enumerate(vals):ax.text(i,v+1000,f'{v:,} ({v/len(raw):.2%})',ha='center',fontsize=10)
ax.set_ylim(0,145000);savefig('01_class_balance.png')

fig,ax=plt.subplots(figsize=(7.4,3.8))
prices=raw['price'].dropna().clip(0,60)
ax.hist(prices,bins=50,color=FOREST,edgecolor='white',linewidth=.35)
ax.set_title('Figure 3. Kindle listing prices (capped at $60 for display)')
ax.set_xlabel('Listed price (USD)');ax.set_ylabel('Books');savefig('02_price_distribution.png')

cats=(raw.groupby('category_name')['isBestSeller'].agg(['mean','size'])
      .query('size>=500').sort_values('mean',ascending=False).head(12))
fig,ax=plt.subplots(figsize=(8.2,5))
ax.barh(cats.index[::-1],(100*cats['mean'])[::-1],color=FOREST)
ax.set_xlabel('Observed bestseller share (%)');ax.set_title('Figure 5. Bestseller incidence by category (n ≥ 500)')
savefig('03_category_rates.png')
cats.rename(columns={'mean':'bestseller_rate','size':'books'}).to_csv(TAB/'02_category_rates.csv')

fig,ax=plt.subplots(figsize=(7.2,3.7))
raw.assign(date_missing=raw['publishedDate'].isna()).groupby('isBestSeller')['date_missing'].mean().mul(100).plot.bar(ax=ax,color=[STONE,FOREST],rot=0)
ax.set_xticklabels(['Not bestseller','Bestseller']);ax.set_ylabel('Missing published date (%)')
ax.set_title('Figure 9. Publication-date missingness by label');savefig('04_date_missingness.png')

fig,ax=plt.subplots(figsize=(7.2,3.8))
for outcome, colour, name in [(False,STONE,'Not bestseller'),(True,FOREST,'Bestseller')]:
    z=np.log1p(raw.loc[raw['isBestSeller']==outcome,'reviews'].clip(lower=0))
    ax.hist(z,bins=44,density=True,alpha=.65,label=name,color=colour)
ax.set_xlabel('log(1 + review count)');ax.set_ylabel('Density')
ax.set_title('Figure 11. Review counts differ by class (excluded from model)');ax.legend()
savefig('05_review_market_signal.png')

print('Figures 1–5 saved; EDA is based on the original, unmodified 133,102 records.')

Figures 1–5 saved; EDA is based on the original, unmodified 133,102 records.


### Expanded descriptive evidence: Figures 12–18
These additional plots distinguish raw bestseller counts from category-specific rates. Title lengths, prices and publication-year distributions are normalised within each class, preventing the 98.3% majority from obscuring shape differences. Kindle Unlimited and recognition flags illustrate platform associations; editor and Goodreads badges are **not** used as model predictors because they may be assigned after market success. Missingness is displayed explicitly so imputation decisions are transparent. Publication-year figures condition on records with usable dates and therefore cannot represent the entire catalogue.

In [5]:
# Expanded descriptive graphics. All counts and percentages are calculated from the loaded CSV.
# Group comparisons are descriptive, not proof that a particular feature causes success.
fig,ax=plt.subplots(figsize=(8.2,4.5))
a=raw.groupby('category_name')['isBestSeller'].agg(['sum','size']).sort_values('sum',ascending=False).head(12)
ax.barh(a.index[::-1],a['sum'].iloc[::-1],color=FOREST)
ax.set_xlabel('Observed bestseller listings');ax.set_title('Figure 6. Categories with most observed bestseller books')
savefig('12_category_positive_counts.png')

fig,ax=plt.subplots(figsize=(7.2,4))
for yv,clr,lab in [(False,STONE,'Not bestseller'),(True,FOREST,'Bestseller')]:
    wc=raw.loc[raw.isBestSeller==yv,'title'].astype(str).str.split().str.len().clip(upper=35)
    ax.hist(wc,bins=np.arange(0,37,2),density=True,color=clr,alpha=.65,label=lab)
ax.set_xlabel('Words in title (display capped at 35)');ax.set_ylabel('Density')
ax.set_title('Figure 7. Book title length distributions by label');ax.legend()
savefig('13_title_word_count.png')

fig,ax=plt.subplots(figsize=(7.2,4))
for yv,clr,lab in [(False,STONE,'Not bestseller'),(True,FOREST,'Bestseller')]:
    prices=raw.loc[raw.isBestSeller==yv,'price'].clip(lower=0,upper=35)
    ax.hist(prices,bins=np.linspace(0,35,36),density=True,color=clr,alpha=.6,label=lab)
ax.set_xlabel('Kindle listing price (USD; values above $35 capped)');ax.set_ylabel('Density')
ax.set_title('Figure 4. Price distributions normalised by outcome');ax.legend()
savefig('14_price_by_label.png')

pd_year=pd.to_datetime(raw['publishedDate'],errors='coerce').dt.year
fig,ax=plt.subplots(figsize=(7.2,4))
for yv,clr,lab in [(False,STONE,'Not bestseller'),(True,FOREST,'Bestseller')]:
    years=pd_year[raw.isBestSeller==yv].dropna().loc[lambda x:x.between(1990,2023)]
    ax.hist(years,bins=np.arange(1990,2025,2),density=True,color=clr,alpha=.6,label=lab)
ax.set_xlabel('Recorded publication year (1990–2023 only)');ax.set_ylabel('Density')
ax.set_title('Figure 8. Publication-year patterns for dated listings');ax.legend()
savefig('15_published_year.png')

ku=(raw.groupby('isKindleUnlimited')['isBestSeller'].agg(['sum','size','mean']).sort_index())
fig,ax=plt.subplots(figsize=(6.6,3.8))
for i,(key,row) in enumerate(ku.iterrows()):
    ax.bar(i,100*row['mean'],color=[STONE,FOREST][i],width=.5)
    ax.text(i,100*row['mean']+.035,f"{100*row['mean']:.2f}%  (n={int(row['size']):,})",ha='center',fontsize=9)
ax.set_xticks(range(len(ku)),['No','Yes']);ax.set_ylabel('Observed bestseller share (%)')
ax.set_xlabel('Kindle Unlimited programme');ax.set_title('Figure 10. Bestseller incidence by Kindle Unlimited status')
ax.set_ylim(0,max(100*ku['mean'])*1.3);savefig('16_kindle_unlimited_rate.png')

flags=raw[['isEditorsPick','isGoodReadsChoice','isBestSeller']]
fig,ax=plt.subplots(figsize=(7.2,3.7))
vals=[100*flags.groupby('isBestSeller')[c].mean().reindex([False,True]).values for c in ['isEditorsPick','isGoodReadsChoice']]
pos=np.arange(2);w=.34
ax.bar(pos-w/2,vals[0],w,label='Editor pick',color=STONE)
ax.bar(pos+w/2,vals[1],w,label='Goodreads choice',color=FOREST)
ax.set_xticks(pos,['Not bestseller','Bestseller']);ax.set_ylabel('Listings with recognition flag (%)')
ax.set_title('Figure 12. Editorial recognition flags (excluded to reduce leakage)');ax.legend()
savefig('17_editorial_flags.png')

miss=raw.isna().sum().sort_values(ascending=False).head(8)
fig,ax=plt.subplots(figsize=(7.2,4.2));ax.barh(miss.index[::-1],miss.values[::-1],color=FOREST)
ax.set_xlabel('Missing values');ax.set_title('Figure 2. Missing entries in original Kindle data')
savefig('18_missing_fields.png')

pandas_spark=raw.groupby('isBestSeller',observed=True).size().rename('books').reset_index()
pandas_spark.to_csv(TAB/'06_target_distribution.csv',index=False)
ku.to_csv(TAB/'07_kindle_unlimited_rates.csv')
print('Expanded EDA figures 12–18 created from actual 133,102 records.')

Expanded EDA figures 12–18 created from actual 133,102 records.


### EDA interpretation
The `isBestSeller` field identifies a small minority of listings, making a majority-only classifier deceptively accurate. `publishedDate` is missing for many records and is imputed **using training data only**. Category-associated differences are descriptive rather than causal. Review counts and star ratings are visualised to demonstrate their potential *contemporaneous* association with success, but are not used as forward-looking predictors; the same applies to editor and Goodreads recognition flags. These features may arise after sales success, creating potential temporal leakage. The data does not identify the original date, rule or sales cutoff underpinning `isBestSeller`.

## 5. Feature selection and leakage controls
**Retained:** listed price, title word count, publication year, category and Kindle Unlimited membership. These are closer to product and distribution descriptors than post-success engagement signals, although recorded snapshots still cannot establish feature timing. **Excluded:** ASIN, URLs and image URL (identifiers), author and seller (high cardinality / availability ambiguity), review count, stars and editorial endorsements (post-outcome leakage risk), `category_id` (duplicates category name). `title_word_count` is preferred to highly correlated raw title characters. The missing publication year is not assumed to be 2023; a missing indicator is automatically created by the imputer.

**Interpretation caveat:** Membership, price and some metadata could still change over a title's lifetime. Thus this is a defensible **cross-sectional classification experiment**, not verified causal or prospective forecast.

In [6]:
work=raw.drop_duplicates(subset=['asin']).copy()
work=work.loc[work['isBestSeller'].notna()].copy()
work['title_word_count']=work['title'].fillna('').astype(str).str.split().str.len()
work['published_year']=pd.to_datetime(work['publishedDate'],errors='coerce').dt.year
# Treat implausible years as missing; do not silently assume they belong to the current year.
work.loc[~work['published_year'].between(1900,2023),'published_year']=np.nan
work['category_name']=work['category_name'].fillna('Unknown').astype(str)
work['isKindleUnlimited']=work['isKindleUnlimited'].astype(str)
# A negative list price cannot be a legitimate sale price; impute it from training observations.
work.loc[~work['price'].between(0,10000),'price']=np.nan
NUM=['price','title_word_count','published_year']
CAT=['category_name','isKindleUnlimited']
X=work[NUM+CAT]
y=work['isBestSeller'].astype(int).to_numpy()
X_train,X_tmp,y_train,y_tmp=train_test_split(X,y,test_size=.30,random_state=SEED,stratify=y)
X_val,X_test,y_val,y_test=train_test_split(X_tmp,y_tmp,test_size=.50,random_state=SEED,stratify=y_tmp)
print('Train/validation/test sizes:',len(y_train),len(y_val),len(y_test))
print('Positive rates:',*[f'{part.mean():.3%}' for part in (y_train,y_val,y_test)])
try:
    encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError:
    encoder=OneHotEncoder(handle_unknown='ignore',sparse=False)
num_pipe=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',encoder)])
preprocess=ColumnTransformer([('numeric',num_pipe,NUM),('categorical',cat_pipe,CAT)],sparse_threshold=0)
Xtr=preprocess.fit_transform(X_train).astype(np.float64)
Xva=preprocess.transform(X_val).astype(np.float64)
Xte=preprocess.transform(X_test).astype(np.float64)
print('Transformed data:',Xtr.shape,Xva.shape,Xte.shape)
assert Xtr.shape[1]==Xva.shape[1]==Xte.shape[1]
assert np.isfinite(Xtr).all() and np.isfinite(Xva).all() and np.isfinite(Xte).all()
feature_names=preprocess.get_feature_names_out().tolist()
pd.DataFrame({'feature':feature_names}).to_csv(TAB/'03_model_features.csv',index=False)

Train/validation/test sizes: 93171 19965 19966
Positive rates: 1.680% 1.678% 1.683%
Transformed data: (93171, 37) (19965, 37) (19966, 37)


## 6. Logistic Regression implemented independently (10 marks)
For an observation \(x_i\), the linear score is \(z_i=x_i^T w + b\). A sigmoid maps this score into \(p_i=1/(1+e^{-z_i})\). Binary cross entropy is the negative mean log likelihood, optionally weighted to offset class imbalance; the gradient of the average weighted loss is \(X^T[s_i(p_i-y_i)]/\sum_i s_i\), and the bias gradient is \(\sum_i s_i(p_i-y_i)/\sum_i s_i\). L2 adds a penalty proportional to \(\|w\|_2^2\) and its corresponding gradient; the bias is not penalised. Gradient descent subtracts the learning-rate-scaled gradient from each parameter. The implementation below uses stable `expit` and `logaddexp` operations, and **does not call an existing estimator's fit function**.

In [7]:
class ScratchLogisticRegression:
    """Original NumPy batch-gradient-descent binary logistic classifier."""
    def __init__(self,lr=.2,epochs=240,l2=0.,balanced=False):
        self.lr=float(lr);self.epochs=int(epochs);self.l2=float(l2);self.balanced=bool(balanced)
    def _loss(self,X,y,weights):
        z=X@self.w+self.b
        # logaddexp(0,z)-y*z is a stable binary cross-entropy formula.
        ce=np.logaddexp(0.,z)-y*z
        return float((weights@ce)/weights.sum() + .5*self.l2*(self.w@self.w))
    def fit(self,X,y,X_valid=None,y_valid=None):
        X=np.asarray(X,dtype=float); y=np.asarray(y,dtype=float)
        self.w=np.zeros(X.shape[1],dtype=float); self.b=0.
        if self.balanced:
            counts=np.bincount(y.astype(int),minlength=2)
            pos=len(y)/(2*counts[1]);neg=len(y)/(2*counts[0])
            weights=np.where(y==1.,pos,neg)
        else:weights=np.ones_like(y)
        self.history={'train':[],'validation':[]}
        for epoch in range(self.epochs):
            probs=expit(X@self.w+self.b)
            residual=weights*(probs-y)
            dw=(X.T@residual)/weights.sum()+self.l2*self.w
            db=residual.sum()/weights.sum()
            self.w-=self.lr*dw
            self.b-=self.lr*db
            if epoch%5==0 or epoch==self.epochs-1:
                self.history['train'].append((epoch+1,self._loss(X,y,weights)))
                if X_valid is not None:
                    self.history['validation'].append((epoch+1,self._loss(X_valid,np.asarray(y_valid,dtype=float),np.ones(len(y_valid)))))
        return self
    def predict_proba(self,X): return expit(np.asarray(X,dtype=float)@self.w+self.b)
    def predict(self,X,threshold=.5):return (self.predict_proba(X)>=threshold).astype(int)

# Analytic-gradient unit check on a small random set (not training our model with sklearn).
rng=np.random.default_rng(SEED)
Xm=rng.normal(size=(20,3)); ym=rng.integers(0,2,20).astype(float)
w=rng.normal(size=3)*.1; b=.1
loss=lambda ww: np.mean(np.logaddexp(0,Xm@ww+b)-ym*(Xm@ww+b))
analytic=Xm.T@(expit(Xm@w+b)-ym)/len(ym)
finite=np.array([(loss(w+np.eye(3)[j]*1e-6)-loss(w-np.eye(3)[j]*1e-6))/(2e-6) for j in range(3)])
assert np.allclose(analytic,finite,atol=1e-6)
print('Gradient finite-difference check: PASSED')

Gradient finite-difference check: PASSED


## 7. Baseline, class weighting, regularisation and validation comparison
The baseline is **not class weighted**; two improvements introduce inverse-prevalence class weighting and optional L2 penalty. The selected model is chosen using validation **F1** and AP (ranking); the classification threshold is selected from validation data rather than from the final test observations. Because F1 depends on the threshold, reported ROC-AUC and AP use continuous probabilities. The baseline majority-only accuracy is explicitly reported for context.

In [8]:
specs={
  'Baseline (unweighted)':dict(lr=.20,epochs=240,l2=.00,balanced=False),
  'Balanced weights':dict(lr=.20,epochs=240,l2=.00,balanced=True),
  'Balanced + L2':dict(lr=.20,epochs=280,l2=.005,balanced=True),
}
models={}; results=[]
threshold_grid=np.linspace(.05,.95,91)
for name,kwargs in specs.items():
    mod=ScratchLogisticRegression(**kwargs).fit(Xtr,y_train,Xva,y_val)
    p=mod.predict_proba(Xva)
    scores=np.array([f1_score(y_val,p>=t,zero_division=0) for t in threshold_grid])
    j=int(scores.argmax()); t=float(threshold_grid[j]); pred=(p>=t).astype(int)
    row={'model':name,'threshold':t,'accuracy':accuracy_score(y_val,pred),
         'precision':precision_score(y_val,pred,zero_division=0),
         'recall':recall_score(y_val,pred,zero_division=0),'f1':f1_score(y_val,pred,zero_division=0),
         'roc_auc':roc_auc_score(y_val,p),'average_precision':average_precision_score(y_val,p)}
    models[name]=mod;results.append(row)
validation=pd.DataFrame(results).sort_values(['f1','average_precision'],ascending=False)
validation.to_csv(TAB/'04_validation_comparison.csv',index=False)
display(validation.round(4))
# Select only from validation data: the final test set remains untouched at this point.
winner=validation.iloc[0].to_dict();best=models[winner['model']]; chosen_threshold=float(winner['threshold'])
print('Validation-selected configuration:',winner['model'],'| threshold:',chosen_threshold)
print('Majority-class validation accuracy:',round((y_val==0).mean(),4))

,model,threshold,accuracy,precision,recall,f1,roc_auc,average_precision
2,Balanced + L2,0.72,0.9094,0.0639,0.3224,0.1067,0.7725,0.0541
1,Balanced weights,0.74,0.9171,0.0646,0.2925,0.1058,0.7711,0.0537
0,Baseline (unweighted),0.05,0.9342,0.0642,0.2149,0.0988,0.7300,0.0464


Validation-selected configuration: Balanced + L2 | threshold: 0.72
Majority-class validation accuracy: 0.9832


### Supplementary validation diagnostics
Figures 19–23 show whether weighting and L2 genuinely improve the held-out validation F1 and average precision. The F1 threshold graph is **validation-only**; this avoids using the final test set to optimise the classification rule. The loss trajectories have different weighting by design and must not be compared as if identical objective functions. Coefficients are conditional on encoding and standardisation, and **cannot be read as causal effects**.

In [9]:
# More model-improvement evidence from validation, without inspecting test labels.
fig,ax=plt.subplots(figsize=(7.7,3.9))
valplot=validation.sort_values('f1',ascending=True)
ax.barh(valplot['model'],valplot['f1'],color=[STONE,OLIVE,FOREST]);ax.set_xlabel('Validation positive-class F1')
ax.set_title('Figure 15. Validation F1 across scratch model configurations')
for i,z in enumerate(valplot['f1']):ax.text(z+.002,i,f'{z:.3f}',va='center')
ax.set_xlim(0,max(valplot['f1'])*1.19);savefig('19_validation_model_f1.png')

fig,ax=plt.subplots(figsize=(7.6,3.9))
valplot=validation.sort_values('average_precision',ascending=True)
ax.barh(valplot['model'],valplot['average_precision'],color=[STONE,OLIVE,FOREST]);ax.set_xlabel('Validation average precision')
ax.set_title('Figure 24. Ranking performance under extreme imbalance')
for i,z in enumerate(valplot['average_precision']):ax.text(z+.0007,i,f'{z:.3f}',va='center')
ax.set_xlim(0,max(valplot['average_precision'])*1.21);savefig('20_validation_model_ap.png')

fig,ax=plt.subplots(figsize=(7.1,4.1))
pval=best.predict_proba(Xva)
vt=np.arange(.05,.951,.01)
vf=np.array([f1_score(y_val,pval>=t,zero_division=0) for t in vt])
ax.plot(vt,vf,color=FOREST,lw=2)
ax.axvline(chosen_threshold,color=RED,linestyle='--',label=f'Selected threshold = {chosen_threshold:.2f}')
ax.set_xlabel('Threshold on validation probabilities');ax.set_ylabel('Validation F1')
ax.set_title('Figure 16. How threshold choice changes F1');ax.legend()
savefig('21_validation_f1_threshold.png')

fig,ax=plt.subplots(figsize=(7.1,4.1))
# Class-weighted training loss differs in definition from the unweighted validation loss.
tr=np.array(best.history['train']);va=np.array(best.history['validation'])
ax.plot(tr[:,0],tr[:,1],color=FOREST,label='Training loss (class weighted + L2)')
ax.plot(va[:,0],va[:,1],color=RED,label='Validation loss (unweighted + L2)')
ax.set_xlabel('Training epoch');ax.set_ylabel('Objective (different weighting)')
ax.set_title('Figure 14. Training convergence and validation monitoring')
ax.legend(fontsize=8);savefig('22_selected_model_losses.png')

coefs=pd.DataFrame({'encoded_feature':feature_names,'weight':best.w})
coefs['absolute_weight']=coefs.weight.abs()
coefs.sort_values('absolute_weight',ascending=False).to_csv(TAB/'08_model_coefficients.csv',index=False)
most=coefs.nlargest(14,'absolute_weight').sort_values('weight')
fig,ax=plt.subplots(figsize=(8.4,5.2))
ax.barh(most.encoded_feature,most.weight,color=[RED if v<0 else FOREST for v in most.weight])
ax.set_xlabel('Model coefficient (encoded/scaled feature; not causal)')
ax.set_title('Figure 25. Largest signed coefficients in selected model')
savefig('23_coefficients.png')
print('Additional validation and coefficient figures 19–23 created.')

Additional validation and coefficient figures 19–23 created.


In [10]:
fig,ax=plt.subplots(figsize=(7.3,4))
for name,model in models.items():
    a=np.array(model.history['train']);b=np.array(model.history['validation'])
    ax.plot(a[:,0],a[:,1],label=f'{name} train',linewidth=1.5)
ax.set_xlabel('Training epoch');ax.set_ylabel('Weighted training cross-entropy + L2')
ax.set_title('Figure 13. Training losses for scratch implementations');ax.legend(fontsize=8)
savefig('06_loss_curves.png')

fig,ax=plt.subplots(figsize=(7.2,4))
for name,model in models.items():
    fpr,tpr,_=roc_curve(y_val,model.predict_proba(Xva))
    ax.plot(fpr,tpr,label=f'{name} (AUC {roc_auc_score(y_val,model.predict_proba(Xva)):.3f})')
ax.plot([0,1],[0,1],linestyle='--',color=STONE,label='Random ranking')
ax.set_xlabel('False positive rate');ax.set_ylabel('True positive rate');ax.set_title('Figure 17. Validation ROC curves')
ax.legend(fontsize=8);savefig('07_validation_roc.png')

fig,ax=plt.subplots(figsize=(7.2,4))
for name,model in models.items():
    pr,rc,_=precision_recall_curve(y_val,model.predict_proba(Xva))
    ax.plot(rc,pr,label=f'{name} (AP {average_precision_score(y_val,model.predict_proba(Xva)):.3f})')
ax.axhline(y_val.mean(),linestyle='--',color=STONE,label='Positive class prevalence')
ax.set_xlabel('Recall');ax.set_ylabel('Precision');ax.set_title('Figure 18. Validation precision–recall curves')
ax.legend(fontsize=8);savefig('08_validation_precision_recall.png')

## 8. Final evaluation: one untouched test set (20 marks)
**False positives** are listings flagged as potential bestsellers that are not labelled as such; they may divert promotional budgets. **False negatives** are labelled bestsellers missed by the model, potentially causing lost opportunities. The publisher should consider the operational cost of each before setting a production threshold. Accuracy alone can be high simply because 98% of listings are negative. PR-AUC / average precision complements ROC-AUC for the rare-positive case.

In [11]:
p_test=best.predict_proba(Xte)
pred_test=(p_test>=chosen_threshold).astype(int)
cm=confusion_matrix(y_test,pred_test,labels=[0,1])
tn,fp,fn,tp=cm.ravel()
final_metrics={
 'selected_model':winner['model'],'validation_chosen_threshold':chosen_threshold,
 'test_n':len(y_test),'test_bestsellers':int(y_test.sum()),
 'test_accuracy':float(accuracy_score(y_test,pred_test)),
 'test_precision':float(precision_score(y_test,pred_test,zero_division=0)),
 'test_recall':float(recall_score(y_test,pred_test,zero_division=0)),
 'test_f1':float(f1_score(y_test,pred_test,zero_division=0)),
 'test_roc_auc':float(roc_auc_score(y_test,p_test)),
 'test_average_precision':float(average_precision_score(y_test,p_test)),
 'test_baseline_majority_accuracy':float((y_test==0).mean()),
 'true_negative':int(tn),'false_positive':int(fp),'false_negative':int(fn),'true_positive':int(tp),
 'spark_executed_in_this_run':SPARK_EXECUTED,'spark_status':SPARK_STATUS,
 'all_records':len(raw),'all_bestsellers':int(raw.isBestSeller.sum()),
 'rows_train':len(y_train),'rows_val':len(y_val),'rows_test':len(y_test),
 'features_used':NUM+CAT,
}
(OUT/'final_metrics.json').write_text(json.dumps(final_metrics,indent=2))
pd.DataFrame([final_metrics]).drop(columns=['features_used']).to_csv(TAB/'05_final_metrics.csv',index=False)
print(json.dumps(final_metrics,indent=2))

fig,ax=plt.subplots(figsize=(5.5,4.6))
im=ax.imshow(cm,cmap='Greys')
for (i,j),v in np.ndenumerate(cm):ax.text(j,i,f'{v:,}',ha='center',va='center',fontsize=12,
  color='white' if v>cm.max()*.50 else INK)
ax.set_xticks([0,1],['Not bestseller','Bestseller']);ax.set_yticks([0,1],['Not bestseller','Bestseller'])
ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.set_title('Figure 19. Final test confusion matrix')
savefig('09_test_confusion_matrix.png')

fig,ax=plt.subplots(figsize=(7.3,4))
fpr,tpr,_=roc_curve(y_test,p_test)
ax.plot(fpr,tpr,color=FOREST,label=f'ROC-AUC {final_metrics["test_roc_auc"]:.3f}')
ax.plot([0,1],[0,1],'--',color=STONE);ax.set_xlabel('False positive rate');ax.set_ylabel('True positive rate')
ax.set_title('Figure 23. Final test ROC curve');ax.legend();savefig('10_test_roc.png')

fig,ax=plt.subplots(figsize=(7.3,4))
p,r,_=precision_recall_curve(y_test,p_test)
ax.plot(r,p,color=FOREST,label=f'AP {final_metrics["test_average_precision"]:.3f}')
ax.axhline(y_test.mean(),linestyle='--',color=STONE,label=f'Prevalence {y_test.mean():.3%}')
ax.set_xlabel('Recall');ax.set_ylabel('Precision');ax.set_title('Figure 20. Final test precision–recall curve')
ax.legend();savefig('11_test_precision_recall.png')

{
  "selected_model": "Balanced + L2",
  "validation_chosen_threshold": 0.72,
  "test_n": 19966,
  "test_bestsellers": 336,
  "test_accuracy": 0.910547931483522,
  "test_precision": 0.060072815533980584,
  "test_recall": 0.29464285714285715,
  "test_f1": 0.09979838709677419,
  "test_roc_auc": 0.7776706874802901,
  "test_average_precision": 0.057504084533679695,
  "test_baseline_majority_accuracy": 0.9831713913653211,
  "true_negative": 18081,
  "false_positive": 1549,
  "false_negative": 237,
  "true_positive": 99,
  "spark_executed_in_this_run": false,
  "spark_status": "NOT EXECUTED: ModuleNotFoundError: No module named 'pyspark'",
  "all_records": 133102,
  "all_bestsellers": 2236,
  "rows_train": 93171,
  "rows_val": 19965,
  "rows_test": 19966,
  "features_used": [
    "price",
    "title_word_count",
    "published_year",
    "category_name",
    "isKindleUnlimited"
  ]
}


### Further held-out test diagnostics
Figures 24–26 illustrate the probability distribution, calibration and precision of the highest-scoring records. Because class weighting changes the loss function, classifier scores should not be assumed to be calibrated probabilities. These are descriptive analyses completed after model and threshold selection, not used to change the selected model.

In [12]:
# Final test figures. Descriptive diagnostics only; do not retune the model on test data.
fig,ax=plt.subplots(figsize=(7.4,4.1))
for v,col,lab in [(0,STONE,'Actual non-bestseller'),(1,FOREST,'Actual bestseller')]:
    ax.hist(p_test[y_test==v],bins=np.linspace(0,1,41),density=True,alpha=.65,color=col,label=lab)
ax.axvline(chosen_threshold,color=RED,ls='--',label='Validation threshold')
ax.set_xlabel('Estimated probability of bestseller status');ax.set_ylabel('Density')
ax.set_title('Figure 26. Test-set predicted-score distributions');ax.legend(fontsize=8)
savefig('24_test_score_distribution.png')

fig,ax=plt.subplots(figsize=(7.4,4.1))
from sklearn.calibration import calibration_curve
frac,mean=calibration_curve(y_test,p_test,n_bins=10,strategy='quantile')
ax.plot(mean,frac,'o-',color=FOREST,label='Observed fraction positive')
ax.plot([0,1],[0,1],'--',color=STONE,label='Perfect agreement')
ax.set_xlim(0,1);ax.set_ylim(0,1);ax.set_xlabel('Mean predicted probability per bin')
ax.set_ylabel('Observed bestseller fraction');ax.set_title('Figure 21. Test reliability diagnostic')
ax.legend(fontsize=8);savefig('25_calibration.png')

order=np.argsort(-p_test)
fractions=[.005,.01,.02,.05,.10,.20]
rows=[]
for frac_size in fractions:
    n=max(1,int(round(len(y_test)*frac_size)))
    precision=float(np.mean(y_test[order[:n]]));lift=precision/float(y_test.mean())
    rows.append((frac_size,n,precision,lift))
lift_tab=pd.DataFrame(rows,columns=['top_fraction','top_n','precision','lift_vs_prevalence'])
lift_tab.to_csv(TAB/'09_test_top_k_diagnostics.csv',index=False)
fig,ax=plt.subplots(figsize=(7.2,3.9))
ax.plot(100*lift_tab['top_fraction'],100*lift_tab['precision'],'o-',color=FOREST,label='Precision among top-ranked listings')
ax.axhline(y_test.mean()*100,linestyle='--',color=RED,label=f'Prevalence {y_test.mean():.2%}')
ax.set_xlabel('Top-scored fraction of test listings (%)');ax.set_ylabel('Observed bestseller prevalence (%)')
ax.set_title('Figure 22. Descriptive precision in highest-scoring test groups')
ax.legend(fontsize=8);savefig('26_test_topk_precision.png')
print('Final diagnostic figures 24–26 saved; no test-set threshold tuning was done.')

Final diagnostic figures 24–26 saved; no test-set threshold tuning was done.


## 9. Discussion, deployment limits and business conclusion
This notebook produces **measured** classification metrics on the supplied dataset, not claimed profits or future sales forecasts. The dataset does not define the bestseller tag's sales threshold or observation date, which restricts external validity. The very low base rate makes false positives costly even when ROC-AUC is acceptable; compare precision with the publisher's required return on marketing effort. Category and pricing associations are observational and must not be interpreted as causal effects. Before deployment, gather historical timestamped book features at or before publication and a fixed forward outcome (e.g., top-sales status within 90 days). Reassess class imbalance, calibrate scores, test across release cohorts and monitor changes over time. No changes were made using test labels after evaluation.

**Spark status for this specific execution:** The notebook prints the exact `SPARK_STATUS` above; if it says not executed, rerun in PySpark-enabled Jupyter/Colab and save the updated executed notebook before final submission.

## 10. Source acknowledgements and references (Harvard Anglia)
- asaniczka (2023) *Amazon Kindle Books Dataset 2023 – 130K Books*. Kaggle. Available at: https://www.kaggle.com/datasets/asaniczka/amazon-kindle-books-dataset-2023-130k-books (Accessed: 13 October 2026).
- Apache Software Foundation (2026) *pyspark.sql.DataFrameReader.csv*. Available at: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html (Accessed: 13 October 2026).
- Pedregosa, F. et al. (2011) ‘Scikit-learn: Machine learning in Python’, *Journal of Machine Learning Research*, 12, pp. 2825–2830. Available at: https://jmlr.org/papers/v12/pedregosa11a.html (Accessed: 13 October 2026).
- Saito, T. and Rehmsmeier, M. (2015) ‘The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets’, *PLOS ONE*, 10(3), e0118432. Available at: https://doi.org/10.1371/journal.pone.0118432 (Accessed: 13 October 2026).
- King, G. and Zeng, L. (2001) ‘Logistic regression in rare events data’, *Political Analysis*, 9(2), pp. 137–163. Available at: https://doi.org/10.1093/oxfordjournals.pan.a004868 (Accessed: 13 October 2026).

**Code declaration:** The gradient-descent class, experiment orchestration and figures in this notebook were independently drafted for this assessment with AI assistance and should be reviewed, tested and disclosed in accordance with institutional rules. Scikit-learn is used only for splits, transformations and evaluation; no ready-made logistic estimator is fitted.

## 11. Final verification and submission checklist
- The notebook executes the complete NumPy classifier and publishes exact figures and metrics from the supplied CSV.
- The independent gradient check must print `PASSED`.
- Confirm `SPARK_EXECUTED == True` **on the submitted notebook**, after running the Spark EDA cell on a computer with Java and PySpark installed. This separate external verification has **not been completed in the current build environment**.
- The `part2_outputs/figures` and `part2_outputs/tables` folders should contain the visual and tabular evidence cited in the report.
- This is a cross-sectional **bestseller-status classification**, not a validated prospective publication forecast.